<img src="../assets/Logo_de_la_Facultad_Experimental_de_Ciencia_y_Tecnología.svg" alt="Facultad Experimental de Ciencia y Tecnologia" width="140">

# Universidad de Carabobo
### Facultad Experimental de Ciencia y Tecnologia (FACYT) — Departamento de Computacion
### Elective: Aprendizaje Automatico — Project Assignment 2 (Data Preparation and Feature Engineering)

**Students:** *Andrés Gubaira & Jhosber Ynojosa*
**Date:** October 2026

---

# Data Preparation and Feature Engineering — Rick and Morty Speaker Prediction

**Purpose.** The EDA (`notebooks/01_eda.ipynb`) identified the problems and hypotheses of this dataset. This
notebook turns them into one reproducible preparation process that is safe against data leakage:

```text
Raw transcript lines
    ↓  validation and row-level filters (rules that learn nothing)
Train / test split, grouped by episode
    ↓
Feature engineering (stateless text features + optional name masking)
    ↓
TF-IDF for the text + scaling for the numeric features
    ↓
Future model (next notebook)
```

This notebook does **not** compare models, tune hyperparameters, select a model or evaluate anything on the test
set. Those steps belong to the modeling notebook.

> **Test-set limitation.** The EDA inspected the complete labelled dataset before this split existed, so the
> test partition created here has already influenced some decisions indirectly (for example, the choice of the
> five main characters). It is a held-out set for the course workflow, not a perfectly untouched final test.
> From this point on, no decision uses the test partition.

## 1. Problem and Context

| Element | Definition in this project |
|---|---|
| **Unit of observation** | One attributed line of dialogue: a block of text assigned to one speaker in one episode. |
| **Target** | The character who said the line (`speaker`, normalized). Baseline definition: one of the five main characters (Rick, Morty, Jerry, Beth, Summer), chosen from the training partition (Section 6.3). |
| **What a prediction means** | Given a new line of dialogue, the model names which of the five characters most likely said it. |
| **Information available when predicting** | Only the text of that single line, and anything computed from that text alone. |
| **Columns excluded, and why** | `Unnamed: 0` (renamed `line_id`): a row counter with no meaning about the speaker. `episode no.`: a new line does not come with its episode, so it is not an input; it is kept only to group the train/test split and the future cross-validation folds. Raw `speaker`: it is the answer itself. Raw `dialogue`: replaced by its cleaned version. Any position, neighbouring line or episode statistic: not available for a single line. |

This is a **multiclass text classification** problem: each line has exactly one speaker, chosen among five
classes.

## 2. Relevant EDA Findings

Only the findings that drive a preparation decision are repeated here (section numbers refer to `01_eda.ipynb`):

- **Long-tail target (Sections 6, 7, 9.1).** 976 raw speaker labels collapse to 840 after formatting cleanup; 44%
  of the speakers have a single line. The five main characters have between 659 and 2,087 lines each, while the
  sixth (Pickle Rick) has 132, almost all from one episode.
- **Episode effects (Sections 3.1, 8, 9.6).** Only 39 of 51 episode numbers are present. Lines of the same
  episode share plot-specific vocabulary (*pluto*, *apples*, *needful*), the share of narration depends on each
  episode's transcriber (0% to 72%), and the 394 exact duplicate rows repeat inside an episode.
- **Noisy text (Sections 5, 9.2).** About 93% of the lines carry layout artifacts (a stray leading `:`,
  newlines, padding, HTML tags) and about 14% contain narration (`[Burps]`, `(laughter)`, lowercase actions).
  174 lines contain no spoken words at all once cleaned.
- **Length and punctuation (Sections 6, 9.4).** Lines are short and right-skewed (median 9 words, 99th
  percentile around 65); `?` and `!` rates differ modestly between speakers.
- **Vocabulary (Sections 9.5 to 9.7).** Distinctive terms include two-word expressions (*oh geez*, *oh man*) and
  function words; about 29% of the lines name a character, which reflects who is being addressed rather than how
  the speaker talks.

## 3. From EDA Findings to Preparation Decisions

| EDA finding | Evidence | Decision | Justification |
|---|---|---|---|
| Target with 840 classes and a long tail | 44% of speakers have 1 line; top 5 have 659 to 2,087 lines, the 6th has 132 (EDA 6, 7, 9.1) | Target = the 5 most frequent speakers, **computed on train only**; other lines are left out | Only these classes have enough examples to learn a character's voice |
| Lines of an episode share plot words and duplicates | *pluto*, *apples*, *needful* occur in one episode; 394 duplicate rows inside episodes (EDA 5, 8, 9.6) | Train/test split **grouped by episode**: whole episodes drawn at random | A random split would let the model memorize an episode's vocabulary and look better than it is |
| `episode no.` is not part of a new line | Prediction-time information set (EDA 1, 9.3) | Excluded as a feature; kept only as the grouping key | A new line does not come with its episode |
| 3 rows without a speaker | Missing-value table (EDA 5) | Drop them | The target cannot be imputed: guessing the speaker would invent labels |
| 174 lines with no spoken text after cleaning | Cleaning results (EDA 9.2) | Drop them before the split | There is nothing to predict from; the rule is applied row by row and learns nothing |
| Speaker labels with formatting noise | 976 raw labels become 840 after removing HTML, colons and spaces (EDA 5) | Deterministic label normalization | `"Rick:"` and `"Rick"` are the same character |
| Variant labels of the main characters | Numbered variants flagged in EDA 11; 20 unambiguous variants with 93 lines (Section 5.3) | Merge only variants that are the same character with a transcriber's note (*Rick (C-137)*, *Morty Voiceover*) | They are the same voice; transformed or alternate versions (*Pickle Rick*, *Toxic Rick*) stay separate |
| Layout artifacts in about 93% of lines, HTML in 304 lines | Marker table (EDA 5) | `clean_dialogue` inside the pipeline | They carry no information about the speaker and would add meaningless tokens (*span*, *style*) |
| Narration in about 14% of lines, density depends on the transcriber | 0% to 72% per episode (EDA 5, 8) | Narration removed in `clean_dialogue`; no narration flag is created | The model should learn what characters say, not each transcriber's style |
| Length is right-skewed | Median 9 words, 99th percentile about 65, long monologues (EDA 6) | `log1p(word count)` | Reduces the skew without removing long lines |
| `?` and `!` rates differ by speaker; a few lines repeat marks (`!!!!!!!!!!`) | Ratio chart (EDA 9.4); repeated marks in 9 training lines (Section 7.1) | `question_ratio`, `exclamation_ratio` on spoken text, a run of marks counted once | Punctuation style is part of the voice; counting runs once keeps 9 lines from dominating the feature |
| Distinctive vocabulary includes two-word expressions and function words | *oh geez*, *oh man*; stop words informative; contractions split into *ll* (EDA 9.6) | TF-IDF of words and two-word expressions, stop words kept, contractions kept whole | Captures catchphrases and function-word habits |
| Names reflect who is addressed, not voice | 29% of lines name a character; 30% of Rick's lines say "Morty" (EDA 9.5) | Name-masking switch, **off** by default, names learned from training labels | Keeps the comparison "voice versus context" available without deciding it here |
| Interjection rates and name-mention flags | Filler profiles (EDA 9.7), mention rates (EDA 9.5) | Not created as separate columns | These words already appear as TF-IDF terms; separate columns would repeat the same information |

Alternatives to these decisions are not discarded: they are listed as hypotheses for the modeling notebook in
Section 12.

## 4. Imports, Constants and Paths

All dependencies come from `requirements.txt`. The random seed fixes which episodes are drawn for the test set
(Section 6.1), so every run produces the same partitions. `EPISODE_RANGE` and
`MAIN_FAMILY` record two facts from the EDA and the show used by the validations. No warning filter is used: if a
library warns, the warning stays visible.

In [1]:
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

RANDOM_STATE = 42
TEST_EPISODE_SHARE = 0.20  # share of episodes drawn at random for the test set
N_TOP_SPEAKERS = 5

ID_COLUMN = "line_id"
GROUP_COLUMN = "episode no."
TEXT_COLUMN = "dialogue"
TARGET_COLUMN = "speaker"
EPISODE_RANGE = (1, 51)  # episode numbers documented by the source transcripts (EDA, Section 3.1)
MAIN_FAMILY = ["Rick", "Morty", "Jerry", "Beth", "Summer"]  # domain knowledge: the show's central family

# Allow execution from the project root or from notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "Rick-n-Morty.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "Rick-n-Morty.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Run this notebook from the repository root or from notebooks/."
    )

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Dataset: {DATA_PATH.relative_to(PROJECT_ROOT).as_posix()}")

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 140)

Python: 3.14.3
pandas: 3.0.2
scikit-learn: 1.9.0
Dataset: data/Rick-n-Morty.csv


## 5. Loading and Validating the Data

### 5.1 Schema contract

**Motivation.** The EDA described the expected columns and values. These checks turn that description into
rules: if the file changes (a missing column, a repeated identifier, an impossible episode number, a missing
dialogue, an infinite value), the notebook stops with a message that explains the problem instead of continuing
silently. The raw column names are also mapped to clearer ones (`dialouge` is a typo in the source file).

In [2]:
RAW_COLUMNS = {"Unnamed: 0": ID_COLUMN, "episode no.": GROUP_COLUMN, "speaker": TARGET_COLUMN, "dialouge": TEXT_COLUMN}

raw = pd.read_csv(DATA_PATH)

missing_columns = [column for column in RAW_COLUMNS if column not in raw.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}. Found: {raw.columns.tolist()}")

unexpected_columns = [column for column in raw.columns if column not in RAW_COLUMNS]
if unexpected_columns:
    raise ValueError(f"Unexpected columns (the schema changed): {unexpected_columns}")

df = raw.rename(columns=RAW_COLUMNS)

if df[ID_COLUMN].duplicated().any():
    raise ValueError(f"{ID_COLUMN} must be unique; found {df[ID_COLUMN].duplicated().sum()} repeated values.")

if not pd.api.types.is_integer_dtype(df[GROUP_COLUMN]):
    raise ValueError(f"{GROUP_COLUMN} must be an integer episode number, got dtype {df[GROUP_COLUMN].dtype}.")

out_of_range = ~df[GROUP_COLUMN].between(*EPISODE_RANGE)
if out_of_range.any():
    raise ValueError(
        f"{GROUP_COLUMN} must be between {EPISODE_RANGE[0]} and {EPISODE_RANGE[1]}; "
        f"found {sorted(df.loc[out_of_range, GROUP_COLUMN].unique())}."
    )

if df[TEXT_COLUMN].isna().any():
    raise ValueError(f"{TEXT_COLUMN} has {df[TEXT_COLUMN].isna().sum()} missing values; every row must carry text.")

if not df[TEXT_COLUMN].map(lambda value: isinstance(value, str)).all():
    raise ValueError(f"{TEXT_COLUMN} must contain only strings.")

numeric_values = df.select_dtypes(include="number")
if not np.isfinite(numeric_values.to_numpy(dtype=float)).all():
    raise ValueError(f"Infinite or missing values in numeric columns: {numeric_values.columns.tolist()}.")

print(f"Schema validated: {df.shape[0]:,} rows, columns {df.columns.tolist()}")
print(f"Episode numbers: {df[GROUP_COLUMN].min()} to {df[GROUP_COLUMN].max()} ({df[GROUP_COLUMN].nunique()} distinct)")
print(f"Target present: {df[TARGET_COLUMN].notna().sum():,} labelled rows, {df[TARGET_COLUMN].isna().sum()} without a speaker")

Schema validated: 9,618 rows, columns ['line_id', 'episode no.', 'speaker', 'dialogue']
Episode numbers: 1 to 51 (39 distinct)
Target present: 9,615 labelled rows, 3 without a speaker


**Interpretation.** The file matches the expected schema: 9,618 rows, a unique identifier, integer episode numbers
between 1 and 51 (39 distinct values, as found in the EDA), text in every row and no infinite values. The target
column is present; 3 rows have no speaker, which is handled below rather than treated as a schema error.

### 5.2 Data quality report

**Motivation.** Some problems are not errors that should stop the notebook, but they must be visible because a
later decision depends on them: missing targets, formatting variants of the same speaker, labels that are not
characters, HTML left by the scraper and duplicated rows. Two of them are hard checks: every main character must
exist as a label after normalization, and no scene header may be normalized into one of them.
`normalize_speaker` is a fixed rule applied to each label on its own (it learns nothing from other rows), so
applying it to the whole file does not leak anything.

In [3]:
HTML_TAG_RE = re.compile(r"<[a-zA-Z/][^>]*>")


def normalize_speaker(label):
    """Deterministic label cleanup: remove HTML, non-breaking spaces, stray colons; unify capitalization."""
    if not isinstance(label, str):
        return np.nan
    name = HTML_TAG_RE.sub("", label).replace("\xa0", " ").strip().rstrip(":").strip()
    return name.title() if name else np.nan


speaker_clean = df[TARGET_COLUMN].map(normalize_speaker)
scene_headers = speaker_clean.dropna().loc[lambda s: s.str.contains("scene", case=False)]

missing_family = [name for name in MAIN_FAMILY if name not in set(speaker_clean)]
if missing_family:
    raise ValueError(f"Main characters missing from the speaker labels after normalization: {missing_family}")
if set(scene_headers) & set(MAIN_FAMILY):
    raise ValueError("A scene header was normalized into a main-character label.")

quality_report = pd.Series({
    "rows without a speaker (target missing)": df[TARGET_COLUMN].isna().sum(),
    "raw speaker labels": df[TARGET_COLUMN].nunique(),
    "speaker labels after formatting cleanup": speaker_clean.nunique(),
    "speaker labels that are scene headers, not characters": scene_headers.nunique(),
    "speaker labels wrapped in HTML": df[TARGET_COLUMN].str.contains(HTML_TAG_RE, na=False).sum(),
    "dialogue values containing HTML": df[TEXT_COLUMN].str.contains(HTML_TAG_RE).sum(),
    "exact duplicate rows (episode, speaker, dialogue)": df.duplicated(subset=[GROUP_COLUMN, TARGET_COLUMN, TEXT_COLUMN]).sum(),
}, name="count").to_frame()
print("Scene-header labels:", sorted(scene_headers.unique()))
quality_report

Scene-header labels: ['(Post-Credit Scene', '--After-Credit Scene--']


,count
rows without a speaker (target missing),3
raw speaker labels,976
speaker labels after formatting cleanup,840
"speaker labels that are scene headers, not characters",2
speaker labels wrapped in HTML,29
dialogue values containing HTML,304
"exact duplicate rows (episode, speaker, dialogue)",394


**Interpretation.** The numbers reproduce the EDA: 976 raw labels become 840 once HTML, non-breaking spaces,
trailing colons and capitalization variants are unified, and 2 labels are scene headers rather than characters.
None of these labels belongs to the five main characters, so they never become a target class. The 394 duplicate
rows are kept: most are short genuine repeats (*"What?"*), and the episode-grouped split of Section 6 keeps every
copy on the same side, so a duplicate cannot appear in both training and test.

### 5.3 Main-character label variants

**Motivation.** The EDA warned that some speaker labels are variants of a main character (*Summer 1*,
*Morty 2*) and should be reviewed. Labels that contain a main character's name are of three kinds:

1. **The same character with a note added by the transcriber:** the home dimension *(C-137)*, *Voiceover*,
   *(On Tv)*, *(Phone)*, or a delivery note such as *(Sarcastic)*.
2. **The character transformed:** *Pickle Rick*, *Tiny Rick*.
3. **A different character or an alternate-universe version:** *Toxic Rick*, *President Morty*, *Morty Jr*,
   *Rick 1*, *Rick D716*.

Only the first kind is unambiguous, so only it is merged into the main character. The rule is fixed: the main name
followed by *Voiceover*, by *(C-137)*, or by a parenthesized note written in ordinary words (dimension codes such as
*(D-99)* are written in capitals and are left out). The other two kinds stay as separate speakers, because the show
presents them as different voices. The rule is applied label by label and learns nothing from the data.

In [4]:
MAIN_CHARACTER_VARIANT = re.compile(
    r"^(?P<name>" + "|".join(MAIN_FAMILY) + r")"
    r"(?:\s+Voiceover|\s*:?\s*\((?:C-137|[^)]*[a-z][^)]*)\).*)$"
)


def canonical_speaker(name):
    """Map unambiguous variants of a main character (C-137, voiceover, on TV, delivery notes) to that character."""
    if not isinstance(name, str):
        return np.nan
    match = MAIN_CHARACTER_VARIANT.match(name)
    return match["name"] if match else name


speaker_canonical = speaker_clean.map(canonical_speaker)
merged = speaker_clean[speaker_clean.notna() & speaker_canonical.ne(speaker_clean)]
merged_table = (
    pd.DataFrame({"mapped_to": speaker_canonical[merged.index], "label": merged})
    .groupby(["mapped_to", "label"]).size().rename("lines").reset_index()
    .sort_values(["mapped_to", "lines"], ascending=[True, False])
    .set_index(["mapped_to", "label"])
)
print(f"{merged.nunique()} variant labels ({len(merged)} lines) merged into a main character; "
      f"speaker labels: {speaker_clean.nunique()} -> {speaker_canonical.nunique()}")
merged_table

20 variant labels (93 lines) merged into a main character; speaker labels: 840 -> 820


lines
mapped_to label                                                                                  
Beth      Beth (On Device)                                                                      1
Jerry     Jerry (C-137)                                                                         4
          Jerry (On Tv)                                                                         3
          Jerry (Kisses Beth)                                                                   1
          Jerry (On Device)                                                                     1
          Jerry: (Awkwardly Glancing At Her Body)                                               1
Morty     Morty Voiceover                                                                      19
          Morty (C-137)                                                                        11
          Morty ( The Same Time With Rick)                                                      1
          Morty (Confused And A Little Concerned)                                               1
          Morty (Gets It)                                                                       1
Rick      Rick (C-137)                                                                         21
          Rick (Phone)                                                                          3
          Rick (Sarcastic)                                                                      2
          Rick (C-137): As You Know, Morty, I'Ve Got A Lotta Enemies In The Universe Th...      1
          Rick (Indifferent)                                                                    1
          Rick (The Same Time With Morty)                                                       1
          Rick (Voiceover)                                                                      1
Summer    Summer Voiceover                                                                     18
          Summer (C-137)                                                                        1

**Interpretation.** 20 labels (93 lines) are merged, and the number of speaker labels goes from 840 to
820. The largest groups are *Rick (C-137)* (21 lines), *Morty Voiceover* (19) and *Summer Voiceover* (18). One
merged label, *Rick (C-137): As You Know, Morty, ...*, contains a whole sentence: a scraping error that attached
text to the name, which the rule still assigns to Rick. Transformed and alternate versions (*Pickle Rick*,
*Toxic Rick*, *Rick 1*) are left untouched.

### 5.4 Spoken-text cleaning and row-level filters

**Motivation.** `clean_dialogue` keeps only the words a character says. It applies the cleaning developed in the
EDA (Section 9.2): it removes HTML, the stray leading `:`, newlines, bracketed or parenthesized narration and, in
the episodes written with the action before the speech, the leading lowercase action. It also removes a
colon left after a sentence end (*"Gross.: For the alien."*, a second `Speaker:` split) and turns curly quotes
(`’`) into straight ones so that *I’m* and *I'm* become the same word. It is a fixed rule applied line by line, so
the same function is used here and, later, inside the pipeline.

Two kinds of rows cannot be used and are removed before the split:

- **No speaker (3 rows):** without a label they can be neither learned from nor evaluated, and imputing a speaker
  would invent the answer.
- **No spoken text after cleaning:** lines that were only narration or scrape residue (`"["`, empty strings). There
  is nothing to predict from.

Both filters look only at the row itself, so they do not use information from other rows or from the test set.

In [5]:
BRACKET_SPAN = re.compile(r"\[[^\[\]]*\]|\([^()]*\)|\*[^*]*\*")
UNOPENED_CLOSE = re.compile(r"^[^()\[\]]*[)\]]")
UNCLOSED_OPEN = re.compile(r"[(\[*][^()\[\]*]*$")
GLUED_WORDS = re.compile(r"(?<=[a-z])(?=[A-Z])")
LEADING_ACTION = re.compile(r"^((?:[a-z][\w'’,\-]*[.!?,;:]?\s+)+?)(?=[A-Z\"“‘'¿¡])")
MID_ACTION = re.compile(r"(?<=[.!?])\s+((?:[a-z][\w'’,\-]*[.!?,;:]?\s+)+?)(?=[A-Z])")
CURLY_QUOTES = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'})


def clean_dialogue(text: str) -> str:
    """Return only the spoken words of a raw transcript line (same rules as the EDA, Section 9.2)."""
    transcript_style = text.startswith("\n")
    s = HTML_TAG_RE.sub(" ", text)
    s = re.sub(r"^\s*:\s*", "", s)
    s = re.sub(r"\s+", " ", s).strip()

    s = BRACKET_SPAN.sub(" ", s)
    s = UNOPENED_CLOSE.sub(" ", s)
    s = UNCLOSED_OPEN.sub(" ", s)

    if transcript_style:
        first, _, rest = s.partition(" ")
        s = (GLUED_WORDS.sub(" ", first) + " " + rest).strip()
        s = LEADING_ACTION.sub(" ", s, count=1)
        s = MID_ACTION.sub(" ", s)

    s = re.sub(r"([.!?])\s*:\s*", r"\1 ", s)  # "Gross.: For the alien." -> "Gross. For the alien."
    s = re.sub(r"\s+([,.!?;:])", r"\1", s)
    s = re.sub(r"[,;:]+([!?.])", r"\1", s)
    s = re.sub(r"^[,.;:\s]+|\s+", lambda m: "" if m.start() == 0 else " ", s).strip()
    return s.translate(CURLY_QUOTES)

In [6]:
df["speaker_clean"] = speaker_canonical
has_target = df["speaker_clean"].notna()
has_spoken_text = df[TEXT_COLUMN].map(clean_dialogue).ne("")

row_filter_summary = pd.DataFrame({
    "rows": [len(df), (~has_target).sum(), (has_target & ~has_spoken_text).sum(), (has_target & has_spoken_text).sum()],
}, index=["raw rows", "dropped: no speaker", "dropped: no spoken text after cleaning", "kept for modeling"])

data = df.loc[has_target & has_spoken_text].reset_index(drop=True)
assert len(data) == row_filter_summary.loc["kept for modeling", "rows"]
row_filter_summary

,rows
raw rows,9618
dropped: no speaker,3
dropped: no spoken text after cleaning,174
kept for modeling,9441


**Interpretation.** 177 rows are removed (3 without a speaker and 174 without spoken words, the same 174 found in
the EDA), leaving 9,441 usable lines. No row is removed for being short or long: one-word lines (*"What?"*) and
long monologues are legitimate dialogue.

## 6. Train/Test Split

### 6.1 Episode-grouped split

**Motivation.** The split happens **before** any transformation that learns from data (vocabulary, IDF weights,
scaler statistics, name list). Rows are not split one by one: whole episodes go either to training or to test.
The EDA showed that an episode's lines share plot vocabulary and duplicates; with a random row split, the model
could memorize *"pluto means Jerry"* from training lines and be rewarded for it on test lines of the same episode.
Holding out whole episodes simulates the real use: lines from episodes the model has never seen.

`GroupShuffleSplit` draws 20% of the episodes **at random** for the test set, with the seed fixed in Section 4. The
draw looks only at episode numbers, never at the labels, so it cannot favour any speaker. A method that also
balances speaker proportions between partitions (`StratifiedGroupKFold`) was considered and not used: it assigns
episodes with a fixed rule that processes the largest, most unbalanced episodes first and uses the seed only to
break ties, so on this data the seed does not change which episodes end up in the test set, and that test set is
not a random sample of episodes. A random draw of episodes represents the whole show better and is easier to
audit.

In [7]:
X = data[[ID_COLUMN, GROUP_COLUMN, TEXT_COLUMN]]
y_all = data["speaker_clean"]

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_EPISODE_SHARE, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, groups=X[GROUP_COLUMN]))

X_train_all, X_test_all = X.iloc[train_idx], X.iloc[test_idx]
y_train_all, y_test_all = y_all.iloc[train_idx], y_all.iloc[test_idx]

print(f"Train: {len(X_train_all):,} lines from {X_train_all[GROUP_COLUMN].nunique()} episodes")
print(f"Test:  {len(X_test_all):,} lines from {X_test_all[GROUP_COLUMN].nunique()} episodes "
      f"({len(X_test_all) / len(X):.1%} of the lines)")
print(f"Test episodes: {sorted(int(episode) for episode in X_test_all[GROUP_COLUMN].unique())}")

Train: 7,521 lines from 31 episodes
Test:  1,920 lines from 8 episodes (20.3% of the lines)
Test episodes: [5, 7, 14, 27, 28, 35, 42, 48]


**Interpretation.** 31 episodes (7,521 lines) go to training and 8 episodes (1,920 lines, 20.3%) to test. The test
episodes (5, 7, 14, 27, 28, 35, 42, 48) come from the beginning, middle and end of the episode range. Because
episodes have very different sizes (from 14 to 859 rows in the EDA), drawing 20% of the episodes does not give
exactly 20% of the lines.

**How much does the size of the test set depend on the draw?** With episodes of such different sizes, another
draw could give a much smaller or larger test set. This check repeats the draw with seeds 0 to 199 and reports only
sizes, never labels. `RANDOM_STATE = 42` was fixed before this check and is not changed because of it: choosing the
seed after seeing its result would be choosing the test set.

In [8]:
draw_sizes = []
for seed in range(200):
    _, draw_test = next(
        GroupShuffleSplit(n_splits=1, test_size=TEST_EPISODE_SHARE, random_state=seed).split(X, groups=X[GROUP_COLUMN])
    )
    draw_sizes.append(len(draw_test) / len(X) * 100)
draw_sizes = pd.Series(draw_sizes, name="test share of lines (%), seeds 0 to 199")

print(f"RANDOM_STATE={RANDOM_STATE}: {len(X_test_all) / len(X):.1%} of the lines, "
      f"which is percentile {(draw_sizes < len(X_test_all) / len(X) * 100).mean():.0%} of the 200 draws")
draw_sizes.describe(percentiles=[.1, .5, .9]).round(1).to_frame()

RANDOM_STATE=42: 20.3% of the lines, which is percentile 46% of the 200 draws


,"test share of lines (%), seeds 0 to 199"
count,200.0
mean,21.0
std,4.1
min,12.4
10%,16.0
50%,20.6
90%,27.1
max,31.0


**Interpretation.** Depending on the draw, the test set would hold between 12.4% and 31.0% of the lines (16.0% to
27.1% for 80% of the seeds). The seed in use gives 20.3%, close to the median (20.6%), so the test set is neither
unusually small nor unusually large. Results on test will still depend on which 8 episodes were drawn; the
episode-grouped cross-validation on the training data, in the modeling notebook, will show how much performance
varies between groups of episodes.

### 6.2 Partition checks

**Motivation.** Verify that the partitions do not share any line or any episode, and that no row was lost. The
check also counts test lines whose text appears somewhere in training: those are not shared observations, but they
show how often the same words are spoken in different episodes.

In [9]:
shared_ids = set(X_train_all[ID_COLUMN]) & set(X_test_all[ID_COLUMN])
shared_episodes = set(X_train_all[GROUP_COLUMN]) & set(X_test_all[GROUP_COLUMN])
if shared_ids:
    raise ValueError(f"{len(shared_ids)} lines appear in both partitions.")
if shared_episodes:
    raise ValueError(f"Episodes {sorted(shared_episodes)} appear in both partitions.")
assert len(X_train_all) + len(X_test_all) == len(X)

train_texts = set(X_train_all[TEXT_COLUMN].map(clean_dialogue))
repeated_text_in_test = X_test_all[TEXT_COLUMN].map(clean_dialogue).isin(train_texts)
print("No line and no episode is shared between train and test.")
print(f"Test lines whose spoken text also occurs somewhere in train: {repeated_text_in_test.sum()} "
      f"({repeated_text_in_test.mean():.1%}), e.g. {X_test_all.loc[repeated_text_in_test, TEXT_COLUMN].map(clean_dialogue).value_counts().head(5).index.tolist()}")

No line and no episode is shared between train and test.
Test lines whose spoken text also occurs somewhere in train: 93 (4.8%), e.g. ['Morty!', 'Grandpa!', 'Yeah.', 'Fair enough.', 'Whoa!']


**Interpretation.** No line and no episode is on both sides. 93 test lines (4.8%) repeat a text that also occurs in
training, and they are short, common lines (*"Yeah."*, *"Whoa!"*, *"Morty!"*) that recur across many episodes and
several speakers. This is a property of conversational dialogue, not leakage: the model cannot use them to identify a
specific episode.

### 6.3 Target definition (learned on train only)

**Motivation.** The list of target classes is a decision learned from data, so it is computed **only on the
training partition**: the five speakers with the most training lines. `frame_target` then applies that same list to
any partition. The notebook stops if the five classes differ from the main family. Test labels are used for one
purpose only: applying this train-learned definition to keep the test lines of the five classes. No statistic of the
test labels is computed or shown; only the number of test lines is reported.

`frame_target(..., include_other=True)` produces the alternative definitions (top-5 or top-10 + `"Other"`), kept
ready for the modeling notebook.

In [10]:
def frame_target(X_part, y_part, classes, include_other=False):
    """Apply a target definition learned on train to any partition."""
    if include_other:
        return X_part, y_part.where(y_part.isin(classes), "Other")
    keep = y_part.isin(classes)
    return X_part.loc[keep], y_part.loc[keep]


TOP_SPEAKERS = y_train_all.value_counts().head(N_TOP_SPEAKERS).index.tolist()
if set(TOP_SPEAKERS) != set(MAIN_FAMILY):
    raise ValueError(f"The top-{N_TOP_SPEAKERS} speakers in train {TOP_SPEAKERS} differ from the main family {MAIN_FAMILY}.")

X_train, y_train = frame_target(X_train_all, y_train_all, TOP_SPEAKERS)
X_test, y_test = frame_target(X_test_all, y_test_all, TOP_SPEAKERS)
groups_train = X_train[GROUP_COLUMN]

train_counts = y_train.value_counts()
print(f"Target classes (learned on train): {TOP_SPEAKERS}")
print(f"Train: {len(X_train):,} lines kept ({len(X_train) / len(X_train_all):.1%} of the train partition); "
      f"test: {len(X_test):,} lines.")
_, y_train_other = frame_target(X_train_all, y_train_all, TOP_SPEAKERS, include_other=True)
pd.DataFrame({
    "train_lines": train_counts,
    "train_share_%": (train_counts / train_counts.sum() * 100).round(1),
})

Target classes (learned on train): ['Rick', 'Morty', 'Jerry', 'Beth', 'Summer']
Train: 4,738 lines kept (63.0% of the train partition); test: 1,117 lines.


,train_lines,train_share_%
speaker_clean,,
Rick,1700,35.9
Morty,1239,26.2
Jerry,710,15.0
Beth,574,12.1
Summer,515,10.9


**Interpretation.** The five classes learned on training are Rick, Morty, Jerry, Beth and Summer. The baseline
target keeps 4,738 training lines (63.0% of the training partition) and 1,117 test lines. Classes are imbalanced
(Rick 35.9%, Summer 10.9%) but every class has hundreds of examples (515 to 1,700).

**Majority-class baseline.** The simplest possible "model" always answers the most frequent class. Its accuracy is
the floor any real model must beat, and it is different for each target definition, so models trained on
different definitions can only be compared against their own baseline. It is computed on training labels only.

In [11]:
def describe_framing(name, labels):
    """Classes, rows and majority-class baseline of one target definition (training labels only)."""
    counts = labels.value_counts()
    return {
        "framing": name,
        "n_classes": labels.nunique(),
        "train_rows": len(labels),
        "majority_class": counts.idxmax(),
        "majority_baseline_accuracy_%": round(counts.max() / counts.sum() * 100, 1),
    }


top_10_train = y_train_all.value_counts().head(10).index.tolist()
_, y_train_top10_other = frame_target(X_train_all, y_train_all, top_10_train, include_other=True)

baseline = pd.DataFrame([
    describe_framing("top-5 only (baseline target)", y_train),
    describe_framing("top-5 + Other (alternative)", y_train_other),
    describe_framing("top-10 + Other (alternative)", y_train_top10_other),
]).set_index("framing")
episodes_per_speaker = X_train_all.groupby(y_train_all)[GROUP_COLUMN].nunique()
print("Training episodes in which each top-10 speaker appears:",
      {speaker: int(episodes_per_speaker[speaker]) for speaker in top_10_train})
baseline

Training episodes in which each top-10 speaker appears: {'Rick': 29, 'Morty': 30, 'Jerry': 25, 'Beth': 26, 'Summer': 25, 'Pickle Rick': 1, 'Mr. Goldenfold': 8, 'Dr. Wong': 1, 'Agency Director': 1, 'Zeep': 1}


,n_classes,train_rows,majority_class,majority_baseline_accuracy_%
framing,,,,
top-5 only (baseline target),5,4738,Rick,35.9
top-5 + Other (alternative),6,7521,Other,37.0
top-10 + Other (alternative),11,7521,Other,33.0


**Interpretation.** Always answering *Rick* is right 35.9% of the time on the baseline target. With top-5 +
`"Other"`, always answering `"Other"` is right 37.0% of the time, and with top-10 + `"Other"`, 33.0%. Because
accuracy rewards the large classes, the modeling notebook should also report macro-F1 and per-class recall.

The top-10 definition, which the EDA listed as an option, adds Pickle Rick, Mr. Goldenfold, Dr. Wong, Agency
Director and Zeep. Four of them appear in a single training episode. With an episode-grouped split, a character
from one episode can be learned or evaluated, never both, so these extra classes would mostly test whether the
model recognizes one episode. This supports top-5 as the baseline; top-10 + `"Other"` stays as an alternative to
measure, not to adopt.

## 7. Feature Engineering

### 7.1 Stateless text features

**Motivation.** `add_text_features` computes every feature from the line's own text. It does not learn averages,
vocabularies or frequencies from other rows, so applying it never leaks information between partitions. It is shown
here as an ordinary function for inspection; Section 10 places the same function inside the pipeline.

| Feature | What it represents | Columns used | Why it could help | Available when predicting? | Leakage or redundancy risk |
|---|---|---|---|---|---|
| `dialogue_clean` | The spoken words only | `dialogue` | It is the input for the text representation; removes layout and narration noise | Yes, it comes from the line | Residual unmarked narration (about 2.5% of lines in the EDA) can still carry transcriber style |
| `word_count_log1p` | Line length, log-transformed | `dialogue_clean` | Rick and Morty produce more of the long lines (EDA 8); the log compresses monologues | Yes | Low; weak on its own (similar medians across speakers) |
| `question_ratio` | Question marks per word | `dialogue_clean` | Some characters ask more | Yes | Low; noisy on very short lines |
| `exclamation_ratio` | Exclamation marks per word | `dialogue_clean` | Some characters exclaim more | Yes | Low; noisy on very short lines |

The features come from three families named in the assignment: simple text features, a logarithmic
transformation and ratios. A line with no words would get ratios of 0 (no words means no punctuation), so the
features can never be missing.

In [12]:
# A run such as "!!!!" or "?!?" counts as one mark: repetition is a transcription choice, not more questions.
QUESTION_RE = re.compile(r"\?(?:[!?]*\?)?")
EXCLAMATION_RE = re.compile(r"!(?:[!?]*!)?")


def add_text_features(X_part: pd.DataFrame) -> pd.DataFrame:
    """Stateless features computed from each line's own text; nothing is learned from other rows."""
    result = X_part.copy()
    spoken = result[TEXT_COLUMN].map(clean_dialogue)
    n_words = spoken.str.split().str.len()
    safe_words = n_words.where(n_words > 0, 1)  # an empty line has no punctuation: its ratios are 0

    result["dialogue_clean"] = spoken
    result["word_count_log1p"] = np.log1p(n_words)
    result["question_ratio"] = spoken.str.count(QUESTION_RE) / safe_words
    result["exclamation_ratio"] = spoken.str.count(EXCLAMATION_RE) / safe_words
    return result

**Preview on a few training lines.** This preview is only for checking that each feature means what it should; the
real transformation happens inside the pipeline.

In [13]:
preview = add_text_features(X_train.sample(6, random_state=RANDOM_STATE))
preview[[TEXT_COLUMN, "dialogue_clean", "word_count_log1p", "question_ratio", "exclamation_ratio"]]

,dialogue,dialogue_clean,word_count_log1p,question_ratio,exclamation_ratio
4736,: Is he going to die?\n,Is he going to die?,1.791759,0.200000,0.000000
468,"You're our boy, dawg. Don't even trip.","You're our boy, dawg. Don't even trip.",2.079442,0.000000,0.000000
2799,"Uh, s-see for yourself.\n","Uh, s-see for yourself.",1.609438,0.000000,0.000000
3735,: No! Don't flashback!,No! Don't flashback!,1.386294,0.000000,0.666667
7828,"Wow, that's pretty crazy, Rick.","Wow, that's pretty crazy, Rick.",1.791759,0.000000,0.000000
277,"Hey, wait, hold on a second, Rick. You wouldn't by any chance have some sor...","Hey, wait, hold on a second, Rick. You wouldn't by any chance have some sort...",3.555348,0.029412,0.000000


**Interpretation.** The leading `:` and the trailing newlines disappear, the words stay unchanged, and the
ratios follow the punctuation (*"Is he going to die?"*: 1 question mark over 5 words = 0.2; *"No! Don't
flashback!"*: 2 exclamation marks over 3 words = 0.67).

**Extreme values (training data only).** Long lines and repeated punctuation are the two sources of extreme values.
None of these lines is removed; the table shows how the chosen definitions handle them.

In [14]:
train_features = add_text_features(X_train)
n_words_train = np.expm1(train_features["word_count_log1p"]).round().astype(int)
naive_exclamation = train_features["dialogue_clean"].str.count("!") / n_words_train

distribution = pd.DataFrame({
    "words per line": n_words_train.describe(percentiles=[.5, .99]),
    "log1p(words per line)": train_features["word_count_log1p"].describe(percentiles=[.5, .99]),
    "exclamation_ratio, each '!' counted": naive_exclamation.describe(percentiles=[.5, .99]),
    "exclamation_ratio, runs counted once": train_features["exclamation_ratio"].describe(percentiles=[.5, .99]),
}).T
distribution["skewness"] = [
    series.skew()
    for series in (n_words_train, train_features["word_count_log1p"], naive_exclamation, train_features["exclamation_ratio"])
]
repeated_marks = train_features["dialogue_clean"].str.contains(r"!!|\?\?")
short_lines = n_words_train <= 2
print(f"Training lines with a repeated mark ('!!' or '??'): {repeated_marks.sum()}")
print(f"Training lines with 1 or 2 words: {short_lines.sum():,} ({short_lines.mean():.1%}); "
      f"their exclamation_ratio is 0, 0.5 or 1 in {train_features.loc[short_lines, 'exclamation_ratio'].isin([0, 0.5, 1]).mean():.0%} of cases")
distribution.round(2)

Training lines with a repeated mark ('!!' or '??'): 9
Training lines with 1 or 2 words: 741 (15.6%); their exclamation_ratio is 0, 0.5 or 1 in 100% of cases


,count,mean,std,min,50%,99%,max,skewness
words per line,4738.0,12.39,12.87,1.00,8.0,58.00,116.00,2.54
log1p(words per line),4738.0,2.22,0.87,0.69,2.2,4.08,4.76,0.04
"exclamation_ratio, each '!' counted",4738.0,0.09,0.28,0.00,0.0,1.00,10.00,13.36
"exclamation_ratio, runs counted once",4738.0,0.08,0.21,0.00,0.0,1.00,1.00,3.29


**Interpretation.** The word count is right-skewed (skewness 2.54, maximum 116 words against a median of 8);
`log1p` brings the skewness to about 0 without dropping any line. Counting every `!` would give one line a ratio of
10 (*"AAAA...HHHH!!!!!!!!!!"*) and a skewness of 13.36; counting a run of marks once caps the ratio at 1 and lowers
the skewness to 3.29. Repeated marks appear in only 9 training lines, and how many marks a transcriber typed is a
formatting choice, so this keeps a handful of lines from dominating the scaled feature.

**Very short lines.** 741 training lines (15.6%) have one or two words, and their ratios can only be 0, 0.5 or 1,
so a single mark moves the value a lot. These lines are **not** filtered out: a model in use must also predict
short lines such as *"Dad?"*, and removing them from training would only hide the problem. Instead,
`word_count_log1p` travels with the ratios, so a model can learn that a ratio from a two-word line is weaker
evidence than one from a long line.

### 7.2 Optional name masking (domain knowledge, learned on train)

**Motivation.** Names in a line mostly tell *who is being addressed*: about 29% of the lines name a character, and
30% of Rick's lines contain "Morty" (EDA 9.5). A model can use that to guess correctly without learning how anyone
talks. `CharacterNameMasker` replaces the names of the target characters (and plurals such as *Mortys*) with
`NAME`, so the model can only use the rest of the sentence.

- **Off by default** (`mask_names=False`): the baseline keeps the names, and the masking stays available to measure
  "voice versus context" in the modeling notebook.
- **Learned on train:** the names come from the training labels in `fit()`, never from test. Inside future
  cross-validation, each fold learns its own list.
- **Why reading `y` is safe here:** a transformer that uses the target is a classic source of leakage, so this is
  stated explicitly. `fit()` keeps only the *set of class names* (the five possible answers), never which line has
  which label, and `transform()` never receives `y`. The class names are known by definition before seeing any
  line: they are the question the model answers.
- **Available when predicting:** yes, it only reads the text of the line.
- **Risk:** names of characters that are not target classes (*Jessica*, *Tommy*) are not masked, and because the
  match ignores capitalization, the rare uses of *summer* as the season are masked too.

In [15]:
class CharacterNameMasker(BaseEstimator, TransformerMixin):
    """Optionally replace the names of the target characters with NAME inside dialogue_clean.

    The names are learned in fit() from the training labels only, so inside cross-validation each
    fold learns its own list. With mask_names=False the transformer leaves the text unchanged.
    """

    def __init__(self, mask_names=False, placeholder="NAME"):
        self.mask_names = mask_names
        self.placeholder = placeholder

    def fit(self, X, y=None):
        if self.mask_names and y is None:
            raise ValueError("CharacterNameMasker needs the training labels to learn which names to mask.")
        names = [] if y is None else sorted(set(pd.Series(y)) - {"Other"})
        self.names_ = names
        self.pattern_ = (
            re.compile(r"\b(?:" + "|".join(map(re.escape, names)) + r")s?\b", flags=re.IGNORECASE) if names else None
        )
        return self

    def transform(self, X):
        result = X.copy()
        if self.mask_names and self.pattern_ is not None:
            result["dialogue_clean"] = result["dialogue_clean"].str.replace(self.pattern_, self.placeholder, regex=True)
        return result


masker_demo = CharacterNameMasker(mask_names=True).fit(train_features, y_train)
demo_rows = train_features.loc[train_features["dialogue_clean"].str.contains(masker_demo.pattern_), ["dialogue_clean"]].head(4)
demo_rows["masked (mask_names=True)"] = masker_demo.transform(demo_rows)["dialogue_clean"]
demo_rows["default (mask_names=False)"] = CharacterNameMasker().fit(train_features, y_train).transform(demo_rows[["dialogue_clean"]])["dialogue_clean"]
print(f"Names learned from the training labels: {masker_demo.names_}")
print(f"Train lines containing one of them: {train_features['dialogue_clean'].str.contains(masker_demo.pattern_).mean():.1%}")
demo_rows

Names learned from the training labels: ['Beth', 'Jerry', 'Morty', 'Rick', 'Summer']
Train lines containing one of them: 28.6%


,dialogue_clean,masked (mask_names=True),default (mask_names=False)
0,Morty! You gotta come on. Jus'... you gotta come with me.,NAME! You gotta come on. Jus'... you gotta come with me.,Morty! You gotta come on. Jus'... you gotta come with me.
1,"What, Rick? What's going on?","What, NAME? What's going on?","What, Rick? What's going on?"
2,"I got a surprise for you, Morty.","I got a surprise for you, NAME.","I got a surprise for you, Morty."
4,"Morty's bed Come on, I got a surprise for you. Morty by the ankle Come on, h...","NAME's bed Come on, I got a surprise for you. NAME by the ankle Come on, hur...","Morty's bed Come on, I got a surprise for you. Morty by the ankle Come on, h..."


**Interpretation.** The learned names are exactly the five target classes, and 28.6% of the training lines contain
at least one of them, close to the 29% found in the EDA. With the switch on, *"What, Rick?"* becomes *"What,
NAME?"*; with the default setting the text is unchanged. The fourth example also shows a known cleaning limitation:
*"Morty's bed"* and *"Morty by the ankle"* are leftover stage directions that the heuristic could not separate.

## 8. Variable Groups

**Motivation.** Each group of variables needs a different preparation, so the groups are declared explicitly.
Anything not listed here is dropped before reaching the model.

| Group | Variables | Preparation |
|---|---|---|
| Free text | `dialogue_clean` | TF-IDF of words and two-word expressions |
| Numeric | `word_count_log1p`, `question_ratio`, `exclamation_ratio` | Standard scaling |
| Categorical nominal | none | Not applicable |
| Categorical ordinal | none | Not applicable |
| Binary indicators | none | Not applicable |
| Excluded | `line_id`, `episode no.`, raw `dialogue` (and the target, which is never part of `X`) | Dropped by the `ColumnTransformer` |

**Why there are no categorical or binary variables.** The only categorical-looking columns in the raw data are the
speaker, which is the target, and the episode, which is not available when predicting. All the information the
model receives comes from the text. In this project the role that one-hot encoding plays in tabular data is played
by the text vectorizer: it turns words into columns. Binary flags were considered (narration present, name
mentioned) and rejected in Section 3: the first is a transcriber-style signal and the second repeats words that
TF-IDF already represents.

In [16]:
TEXT_FEATURE = "dialogue_clean"
NUMERIC_FEATURES = ["word_count_log1p", "question_ratio", "exclamation_ratio"]
NOMINAL_FEATURES = []
ORDINAL_FEATURES = []
BINARY_FEATURES = []
EXCLUDED_COLUMNS = {
    ID_COLUMN: "row identifier, carries no information about the speaker",
    GROUP_COLUMN: "not available when only a line is given; kept in X only to group the validation folds",
    TEXT_COLUMN: "raw text with layout and narration; replaced by dialogue_clean",
}

## 9. Pipelines by Variable Type

**Text: `TfidfVectorizer`.**

- It builds a vocabulary from the **training** lines and represents each line as a vector with one column per
  term. TF-IDF gives more weight to terms that are frequent in the line but rare across lines, so very common words
  weigh less than distinctive ones. Each row is normalized to length 1, so long and short lines are comparable.
- `ngram_range=(1, 2)`: single words and two-word expressions, because catchphrases such as *oh geez* are two words
  long (EDA 9.6).
- `token_pattern` keeps contractions whole (*I'll*, *don't*); the default pattern reduces *I'll* to the fragment *ll*
  and *don't* to *don*, a limitation noted in the EDA. Stop words are **not** removed: function words are part of a character's voice.
- `min_df=2`: a term must appear in at least two training lines, because a term seen once cannot generalize. This
  is how rare categories are handled for text: rare terms are dropped instead of getting their own column. Words
  never seen in training are simply ignored when new text is transformed, as `handle_unknown="ignore"` does in
  one-hot encoding.
- These settings are fixed from the EDA evidence and are not tuned here; comparing alternatives belongs to the
  modeling notebook.

**Numeric: `StandardScaler`.** It centers each feature on its training mean and divides by its training standard
deviation, so the three numeric features are on comparable scales and no single one dominates models that are
sensitive to magnitude (linear models, distance-based models).

**No imputer.** No feature can be missing by construction: the schema check in Section 5.1 stops the notebook if a
dialogue is missing, and the ratios are defined as 0 for a line without words. Adding an imputer would hide a future
data problem instead of exposing it.

In [17]:
TOKEN_PATTERN = r"(?u)\b\w[\w']*\b"

text_pipeline = TfidfVectorizer(
    lowercase=True,
    token_pattern=TOKEN_PATTERN,
    ngram_range=(1, 2),
    min_df=2,
)

numeric_pipeline = Pipeline(steps=[("scaler", StandardScaler())])

## 10. `ColumnTransformer` and Complete Pipeline

**Motivation.** The `ColumnTransformer` applies each group's preparation and, with `remainder="drop"`, discards
every column that is not listed, so the excluded columns cannot reach the model. The preparation pipeline chains
the three stages in order: stateless text features, optional name masking, and the column transformations.
Feature engineering therefore lives inside the pipeline instead of depending on manual cells.

`build_model_pipeline(estimator, mask_names=False)` returns an **unfitted** copy of the whole chain with a model at
the end. The modeling notebook will pass raw lines to it, and every learned step (vocabulary, IDF weights, scaler,
name list) will be fitted inside each cross-validation fold. The cell prints the structure of that pipeline, with
`"passthrough"` in place of the estimator, which the modeling notebook will supply.

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_pipeline, TEXT_FEATURE),
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ],
    remainder="drop",
)

preparation_pipeline = Pipeline(steps=[
    ("text_features", FunctionTransformer(add_text_features, validate=False)),
    ("name_masking", CharacterNameMasker(mask_names=False)),
    ("preprocessor", preprocessor),
])


def build_model_pipeline(estimator, mask_names=False) -> Pipeline:
    """Unfitted end-to-end pipeline: raw lines -> features -> TF-IDF + scaling -> estimator."""
    preparation = clone(preparation_pipeline).set_params(name_masking__mask_names=mask_names)
    return Pipeline(steps=[*preparation.steps, ("model", estimator)])


for step_name, step in build_model_pipeline("passthrough").steps:
    print(f"{step_name:<20} {type(step).__name__ if step != 'passthrough' else 'passthrough (estimator supplied later)'}")
for block, transformer, columns in preprocessor.transformers:
    print(f"    {block:<8} {type(transformer).__name__:<16} <- {columns}")

text_features        FunctionTransformer
name_masking         CharacterNameMasker
preprocessor         ColumnTransformer
model                passthrough (estimator supplied later)
    text     TfidfVectorizer  <- dialogue_clean
    numeric  Pipeline         <- ['word_count_log1p', 'question_ratio', 'exclamation_ratio']


## 11. Structural Verification (training data only)

**Motivation.** The pipeline is fitted once on the training partition **only to inspect its output**. This is not
model training: there is no estimator. The test partition is not transformed. A copy of `X_train` is kept to prove
that the pipeline does not modify its input.

In [19]:
X_train_snapshot = X_train.copy(deep=True)

X_train_prepared = preparation_pipeline.fit_transform(X_train, y_train)
output_feature_names = preparation_pipeline.named_steps["preprocessor"].get_feature_names_out()

print(f"Raw training input:      {X_train.shape[0]:,} rows x {X_train.shape[1]} columns {X_train.columns.tolist()}")
print(f"Prepared training matrix: {X_train_prepared.shape[0]:,} rows x {X_train_prepared.shape[1]:,} features "
      f"({type(X_train_prepared).__name__})")

Raw training input:      4,738 rows x 3 columns ['line_id', 'episode no.', 'dialogue']
Prepared training matrix: 4,738 rows x 11,490 features (csr_matrix)


In [20]:
checks = {}
checks["same number of rows as the input"] = X_train_prepared.shape[0] == X_train.shape[0]
checks["one name per output column"] = X_train_prepared.shape[1] == len(output_feature_names)
checks["no NaN or infinite values"] = bool(np.isfinite(X_train_prepared.data).all())
fitted_preprocessor = preparation_pipeline.named_steps["preprocessor"]
columns_used = {
    column
    for name, _, columns in fitted_preprocessor.transformers_
    if name != "remainder"
    for column in ([columns] if isinstance(columns, str) else columns)
}
checks["no transformer reads an excluded column"] = not (columns_used & set(EXCLUDED_COLUMNS))
checks["unlisted columns are dropped (remainder='drop')"] = fitted_preprocessor.remainder == "drop"
checks["every output comes from the text or numeric block"] = all(
    name.startswith(("text__", "numeric__")) for name in output_feature_names
)
try:
    pd.testing.assert_frame_equal(X_train, X_train_snapshot)
    checks["original X_train left unchanged"] = True
except AssertionError:
    checks["original X_train left unchanged"] = False

checks_table = pd.Series(checks, name="passed").to_frame()
if not checks_table["passed"].all():
    raise AssertionError(f"Failed checks: {checks_table.index[~checks_table['passed']].tolist()}")
checks_table

,passed
same number of rows as the input,True
one name per output column,True
no NaN or infinite values,True
no transformer reads an excluded column,True
unlisted columns are dropped (remainder='drop'),True
every output comes from the text or numeric block,True
original X_train left unchanged,True


**Interpretation.** All checks pass: the 4,738 training lines are preserved, every output column has a name, the
matrix contains no missing or infinite values, no transformer reads the identifier, the episode number or the raw
text, every output column comes from the text or numeric block, and `X_train` is identical before and after the
transformation.

**Dimensions and output features.**

In [21]:
feature_blocks = pd.Series([name.split("__", 1)[0] for name in output_feature_names]).value_counts()
vocab = pd.Series(output_feature_names[[name.startswith("text__") for name in output_feature_names]]).str.removeprefix("text__")
n_bigrams = vocab.str.contains(" ").sum()
density = X_train_prepared.nnz / (X_train_prepared.shape[0] * X_train_prepared.shape[1])

print(f"Output columns by block: {feature_blocks.to_dict()}")
print(f"  text block: {len(vocab) - n_bigrams:,} single words + {n_bigrams:,} two-word expressions")
print(f"Non-zero values per line (median): {np.median(np.diff(X_train_prepared.indptr)):.0f} of {X_train_prepared.shape[1]:,}")
print(f"Matrix density: {density:.4%}")
print("Terms containing 'geez':", vocab[vocab.str.contains("geez")].tolist())
print("Numeric output names:", [name for name in output_feature_names if name.startswith("numeric__")])

Output columns by block: {'text': 11487, 'numeric': 3}
  text block: 2,896 single words + 8,591 two-word expressions
Non-zero values per line (median): 16 of 11,490
Matrix density: 0.1846%
Terms containing 'geez': ['aw geez', 'geez', 'geez dad', 'geez frank', 'geez i', 'geez morty', 'geez ohh', 'geez okay', 'geez rick', "geez what's", 'geez you', 'oh geez']
Numeric output names: ['numeric__word_count_log1p', 'numeric__question_ratio', 'numeric__exclamation_ratio']


**Interpretation.** The input has 3 columns and the output has 11,490: 2,896 single words, 8,591 two-word
expressions and the 3 scaled numeric features. That is more columns than training lines, which is normal for text,
but the matrix is very sparse: a typical line activates 16 columns and only 0.18% of the entries are non-zero, so
it is stored as a sparse matrix. The vocabulary keeps the catchphrases found in the EDA, for example *geez*, *oh
geez* and *aw geez*. Two consequences for the modeling notebook: models that handle sparse,
high-dimensional input (linear models, Naive Bayes) are natural candidates, and the large number of two-word terms
seen in only a few lines is a risk of overfitting that the comparison "single words versus words plus pairs" should
measure.

**Check of the name-masking switch.** The alternative configuration is fitted the same way (training data only) to
confirm that, when the switch is on, no term in the vocabulary still contains a target name.

In [22]:
masked_pipeline = clone(preparation_pipeline).set_params(name_masking__mask_names=True)
X_train_masked = masked_pipeline.fit_transform(X_train, y_train)
masked_vocab = masked_pipeline.named_steps["preprocessor"].get_feature_names_out()
name_tokens = {n.lower() for n in TOP_SPEAKERS}
leftover = [n for n in masked_vocab if n.startswith("text__") and set(n.removeprefix("text__").split()) & name_tokens]
if leftover:
    raise AssertionError(f"Name masking left target names in the vocabulary: {leftover[:10]}")
print(f"mask_names=False: {X_train_prepared.shape[1]:,} features")
print(f"mask_names=True:  {X_train_masked.shape[1]:,} features; no term contains a target name "
      f"({len(TOP_SPEAKERS)} names replaced by NAME)")

mask_names=False: 11,490 features
mask_names=True:  11,371 features; no term contains a target name (5 names replaced by NAME)


**Interpretation.** With masking on, the vocabulary shrinks from 11,490 to 11,371 columns and no term contains
*rick*, *morty*, *jerry*, *beth* or *summer*: all of them collapse into the single token *name*.

## 12. Conclusions and Handoff to the Modeling Notebook

**Decisions taken.**
- Target: the five main characters (Rick, Morty, Jerry, Beth, Summer), learned on the training partition. Majority
  baseline on train: 35.9% (Rick).
- Labels: formatting cleanup (976 to 840 labels) and merging of 20 unambiguous variants of the main characters
  (93 lines, 840 to 820 labels).
- Rows: 3 without a speaker and 174 without spoken text removed before the split; no row removed for its length;
  duplicates kept.
- Split: whole episodes drawn at random, 31 for training (7,521 lines, 4,738 of the target classes) and 8 for test
  (1,920 lines, 1,117 of the target classes); no shared line or episode.
- Features: `dialogue_clean` represented with TF-IDF of words and two-word expressions, plus `word_count_log1p`,
  `question_ratio` and `exclamation_ratio`, scaled. Name masking available and off.
- Everything that learns from data (target classes, vocabulary, IDF weights, scaler, name list) is fitted on
  training data only.

**Excluded variables.** `line_id` (identifier), `episode no.` (not available when predicting; only the grouping
key), raw `dialogue` (replaced by its cleaned version), raw `speaker` (target), narration flags (transcriber style)
and name-mention or interjection columns (redundant with TF-IDF).

**Limitations.**
- The EDA saw the whole dataset, so the test set is not perfectly untouched.
- Some unmarked narration remains in the spoken text, as seen in Section 7.2.
- With only 8 test episodes (5, 7, 14, 27, 28, 35, 42, 48), results on test will depend on which episodes were
  drawn; cross-validation grouped by episode on the training data will give a more stable estimate.
- Lines from the other 815 speakers, including transformed or alternate versions of the main characters such as
  *Pickle Rick*, are not used by the baseline target.
- Names of non-target characters are never masked.

**Hypotheses to compare in the modeling notebook.**
1. Top-5 only versus top-5 + `"Other"` versus top-10 + `"Other"`, each against its own baseline (35.9%, 37.0%
   and 33.0%).
2. `mask_names=False` versus `mask_names=True` (voice versus conversational context), and an extended variant that
   also masks address terms (*dad*, *mom*, *grandpa*), which reveal who is being addressed just like names (EDA 9.5).
3. Single words versus words plus two-word expressions.
4. Raw `dialogue` versus `dialogue_clean`.
5. With versus without the three numeric features.
6. Episode-grouped cross-validation versus a random stratified one, to measure how optimistic the latter is.
7. Optional, and reported separately: a contextual model that also receives the episode or the previous lines.
   It answers a different question from this project's text-only premise, so it can never replace the main model.

**Objects for the modeling notebook.** No file is written by this notebook. The modeling notebook will rebuild the
following objects by running the same code of Sections 4 to 6 (constants, validation, cleaning, filters, split and
target) and Sections 7, 9 and 10 (features and pipeline) without changes; any change to that code must be made in
both notebooks, otherwise the partitions would differ:

| Object | Content |
|---|---|
| `X_train`, `y_train` | 4,738 raw training lines (`line_id`, `episode no.`, `dialogue`) and their speaker |
| `groups_train` | Episode of each training line, for `StratifiedGroupKFold` or `GroupKFold` |
| `X_test`, `y_test` | 1,117 locked test lines, to be used once, after the complete pipeline is chosen |
| `TOP_SPEAKERS`, `frame_target` | Target classes and the function that applies any target definition learned on train (top-5, top-5 or top-10 + `"Other"`) |
| `build_model_pipeline` | Unfitted raw-text-to-model pipeline, with the `mask_names` switch |